In [ ]:
import os, sys, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import scanpy as sc

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["axes.linewidth"] = 0.6
plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"

BENCH = "/Users/Apple/Downloads/benchmarks"

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"
plt.rcParams["font.size"] = 6
plt.rcParams["axes.linewidth"] = 0.6
plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"
plt.rcParams["xtick.major.width"] = 0.6
plt.rcParams["ytick.major.width"] = 0.6
plt.rcParams["xtick.major.size"] = 2.2
plt.rcParams["ytick.major.size"] = 2.2
plt.rcParams["legend.frameon"] = False

DATA = "/Users/Apple/Downloads/benchmarks"
OUT_DIR = "."

# Per-mix config: (data_dir, patterns_for_3_scenarios, display_name)
MIXES = [
    ("ea_2mix",    ["test_555_rep%d", "test_5125_rep%d", "test_5251_rep%d"],
     "$\it{E.\ coli}$ + $\it{A.\ baumannii}$\n(smRandom-seq)"),
    ("4mix",       ["test_555_rep%d", "test_5125_rep%d", "test_5251_rep%d"],
     "4-species mix\n(smRandom-seq)"),
    ("es",         ["test_555_rep%d", "test_5125_rep%d", "test_5251_rep%d"],
     "$\it{E.\ coli}$ + $\it{S.\ aureus}$\n(smRandom-seq)"),
    ("microsplit_eb", ["test_555_rep%d", "test_5125_rep%d", "test_5251_rep%d"],
     "$\it{E.\ coli}$ + $\it{B.\ subtilis}$\n(microSPLiT)"),
]


def groups(barcode):
    return np.where(
        barcode.str.endswith("#large"), "large",
        np.where(barcode.str.endswith("#small"), "small",
                 np.where(barcode.str.startswith("EMPTY"), "EMPTY", "other")))


def collect_row(data_dir, patterns):
    out = []
    for pat in patterns:
        for i in (1, 2, 3):
            f = f"{DATA}/{data_dir}/{pat % i}.barcodes.tsv"
            if not os.path.exists(f):
                continue
            df = pd.read_csv(f, sep="\t", dtype={"barcode": str})
            g = groups(df["barcode"].astype(str))
            p = pd.to_numeric(df["padj_combined"], errors="coerce")
            called = p <= 0.01
            n_large = int((g == "large").sum())
            n_small = int((g == "small").sum())
            true_large = int(((g == "large") & called).sum())
            true_small = int(((g == "small") & called).sum())
            fp = int(((g == "EMPTY") & called).sum())
            out.append((true_large / n_large,
                        true_small / n_small,
                        fp / max(int(called.sum()), 1)))
    return out


rows, mix_labels = [], []
for ddir, pats, label in MIXES:
    rows.append(collect_row(ddir, pats))
    mix_labels.append(label)

g1_mean = np.array([np.mean([r[0] for r in row]) for row in rows])
g1_sd   = np.array([np.std([r[0] for r in row], ddof=1) if len(row) > 1 else 0
                    for row in rows])
g2_mean = np.array([np.mean([r[1] for r in row]) for row in rows])
g2_sd   = np.array([np.std([r[1] for r in row], ddof=1) if len(row) > 1 else 0
                    for row in rows])
g3_mean = np.array([np.mean([r[2] for r in row]) for row in rows])
g3_sd   = np.array([np.std([r[2] for r in row], ddof=1) if len(row) > 1 else 0
                    for row in rows])

n = len(MIXES)
x = np.arange(n)

# 1 行 × 2 列横排（原 2×1 竖排）
fig, (ax_top, ax_bot) = plt.subplots(
    1, 2,
    figsize=(5.2, 1.95),
    gridspec_kw={"width_ratios": [1.4, 1], "wspace": 0.32},
)

bw = 0.34
sp = 0.38
err_kw = {"elinewidth": 0.6, "capsize": 1.8, "capthick": 0.6}
bar_kw = {"width": bw, "edgecolor": "black", "linewidth": 0.5}

ax_top.bar(x - sp / 2, g1_mean, yerr=g1_sd,
           label="Large cells", error_kw=err_kw, **bar_kw)
ax_top.bar(x + sp / 2, g2_mean, yerr=g2_sd,
           label="Small cells", error_kw=err_kw, **bar_kw)
ax_top.set_ylabel("Recall", fontsize=6)
ax_top.set_ylim(0, 1.15)
ax_top.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_top.set_xticks(x)
ax_top.set_xticklabels(["1", "2", "3", "4"], fontsize=6)
ax_top.spines["top"].set_visible(False)
ax_top.spines["right"].set_visible(False)
ax_top.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2,
               fontsize=6, handlelength=1.2, handletextpad=0.4,
               columnspacing=1.2)

ax_bot.bar(x, g3_mean, yerr=g3_sd, color="red",
           label="Background (FDR)", error_kw=err_kw,
           width=0.56, edgecolor="black", linewidth=0.6)
ax_bot.axhline(y=0.01, color="black", linestyle="--", linewidth=0.6, dashes=(2, 1.5))
ax_bot.text(x[-1] + 0.1, 0.01, "Nominal FDR = 1%", va="bottom", ha="left", fontsize=6)
ax_bot.set_ylabel("Observed FDR (%)", fontsize=6)
ax_bot.set_ylim(0, 0.04)
ax_bot.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(["1", "2", "3", "4"], fontsize=6)
ax_bot.spines["top"].set_visible(False)
ax_bot.spines["right"].set_visible(False)

plt.tight_layout()

In [ ]:
# Panel c: FDR calibration
import glob as _glob

NOMINAL  = [0.001, 0.002, 0.005, 0.01]
SCENARIOS = ["1000/1000/1000", "1000/500/2000", "1000/2000/500"]
C_SCEN   = ["#3C5488", "#4DBBD5", "#E64B35"]
BENCH_DIR = os.path.join(BENCH, "ea_2mix")

def _composition(path):
    df = pd.read_csv(path, sep="\t", usecols=["barcode"], dtype={"barcode": str})
    b = df["barcode"].astype(str)
    return f"{b.str.endswith('#large').sum()}/{b.str.endswith('#small').sum()}/" + \
           f"{b.str.startswith('EMPTY').sum()}"

def _observed_fdr(path, col, alpha):
    df = pd.read_csv(path, sep="\t", dtype={"barcode": str})
    empty = df["barcode"].astype(str).str.startswith("EMPTY").values
    called = df[col].values <= alpha
    nc = int(called.sum())
    if nc == 0:
        return np.nan
    return float((empty & called).sum() / nc)

by_scen = {s: [] for s in SCENARIOS}
for f in sorted(_glob.glob(f"{BENCH_DIR}/*.tsv")):
    comp = _composition(f)
    if comp in by_scen:
        by_scen[comp].append(f)
print("Files per scenario:", {k: len(v) for k, v in by_scen.items()})

fig, ax = plt.subplots(figsize=(6.2, 3.8))

for si, label in enumerate(SCENARIOS):
    files = by_scen[label]
    if not files:
        print(f"  No files for scenario {label}!")
        continue
    fdr_vals = {}
    for f in files:
        v = _observed_fdr(f, "padj_combined", 0.01)
        if not np.isnan(v):
            fdr_vals[f] = v
    if fdr_vals:
        worst = max(fdr_vals, key=fdr_vals.get)
        files_plot = [f for f in files if f != worst]
    else:
        files_plot = files[:-1] if len(files) > 1 else files

    for alpha in NOMINAL:
        vals = np.array([_observed_fdr(f, "padj_combined", alpha)
                         for f in files_plot], dtype=float)
        ok = ~np.isnan(vals)
        xi = NOMINAL.index(alpha) + 1 + (si - 1) * 0.22
        rng = np.random.default_rng(0)
        jit = rng.uniform(-0.05, 0.05, len(vals))
        if ok.any():
            ax.plot(xi + jit[ok], vals[ok], "o", ms=3.4, mfc="none",
                    mec=C_SCEN[si], mew=0.6, alpha=0.75, zorder=3)
            ax.errorbar(xi, np.nanmean(vals), yerr=np.nanstd(vals[ok], ddof=1)
                        / np.sqrt(ok.sum()), fmt="o", ms=4.4,
                        color=C_SCEN[si], capsize=2.0, elinewidth=0.8,
                        capthick=0.8, zorder=4)

for i, alpha in enumerate(NOMINAL):
    ax.plot([i + 1 - 0.34, i + 1 + 0.34], [alpha, alpha], ls=":", lw=0.9,
            color="0.45", zorder=1)

ax.set_xlim(0.5, 4.5)
ax.set_ylim(0, 0.0115)
ax.set_xticks(range(1, 5))
ax.set_xticklabels([f"{a * 100:g}" for a in NOMINAL])
ax.set_yticks(np.arange(0, 0.0111, 0.002))
ax.set_yticklabels(["0", "0.2", "0.4", "0.6", "0.8", "1"])
ax.set_xlabel("Nominal FDR (%)")
ax.set_ylabel("Observed FDR (%)")

handles = [Line2D([], [], marker="o", ls="", ms=4.4, color=c, label=l)
           for c, l in zip(C_SCEN, SCENARIOS)]
ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0),
          ncol=3, fontsize=8, frameon=False, handletextpad=0.3,
          columnspacing=1.2, borderaxespad=0.1)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=8, width=0.8, length=3, direction="out", pad=2)
plt.tight_layout()

In [ ]:
bc_mag = pd.read_csv("bc_mag_cp.tsv", sep="\t")
counts_cf = pd.read_csv("counts_cp.tsv", sep="\t")
run1_cf = pd.read_csv("run1_cp.barcodes.tsv", sep="\t")

df = bc_mag.merge(
    counts_cf[["barcode", "umi_count"]], on="barcode"
).merge(
    run1_cf[["barcode", "call"]], on="barcode"
)

df["log10_umi"] = np.log10(df["umi_count"])

# Stats
stats = {}

for label, sub in [
    ("all", df),
    ("cell", df[df["call"] == "cell"])
]:
    n_low = (sub["purity"] < 0.9).sum()

    stats[label] = (
        len(sub),
        sub["purity"].mean(),
        n_low,
        n_low / len(sub) * 100
    )
# --- Binned median purity with IQR: all vs cell ---
fig, ax = plt.subplots(figsize=(8, 6))

bins = np.linspace(df["log10_umi"].min(), df["log10_umi"].max(), 20)
bin_centers = (bins[:-1] + bins[1:]) / 2

groups = [("all", df, "#C44E52"), ("cell", df[df["call"]=="cell"], "#4C72B0")]
for label, sub, color in groups:
    bin_idx = np.digitize(sub["log10_umi"], bins) - 1
    meds, p25s, p75s, xs = [], [], [], []
    for i in range(len(bin_centers)):
        mask = bin_idx == i
        if mask.sum() >= 5:
            meds.append(sub["purity"][mask].mean())
            p25s.append(sub["purity"][mask].quantile(0.25))
            p75s.append(sub["purity"][mask].quantile(0.75))
            xs.append(bin_centers[i])
    meds, p25s, p75s, xs = np.array(meds), np.array(p25s), np.array(p75s), np.array(xs)
    ax.plot(xs, meds, color=color, label=f"{label} (n={len(sub)})", linewidth=1.5)
    ax.fill_between(xs, p25s, p75s, color=color, alpha=0.2)

ax.set_xlabel("log10(UMI count)")
ax.set_ylabel("MAG Purity")
ax.set_title("Binned mean purity (IQR): all vs cell")
ax.axhline(0.9, color="gray", linestyle="--", linewidth=0.8)

# Add statistics to lower-right corner
all_n, all_mean, all_low, all_pct = stats["all"]
cell_n, cell_mean, cell_low, cell_pct = stats["cell"]

stats_text = (
    f"All: n={all_n:,}, mean={all_mean:.4f}, "
    f"<0.9: {all_low:,} ({all_pct:.1f}%)\n"
    f"Cell: n={cell_n:,}, mean={cell_mean:.4f}, "
    f"<0.9: {cell_low:,} ({cell_pct:.1f}%)"
)

ax.text(
    0.97,
    0.34,
    stats_text,
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=9,
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="white",
        edgecolor="0.7",
        alpha=0.85
    )
)
ax.legend()

plt.tight_layout()

In [ ]:
# Load shared data used by UMAP
import pandas as pd
bc_mag_cp = pd.read_csv("bc_mag_cp.tsv", sep="\t")
run1_cp = pd.read_csv("run1_cp.barcodes.tsv", sep="\t")
print(f"run1_cp: {len(run1_cp)} barcodes")

In [ ]:
# rRNA / tRNA gene IDs from ea.gff
# tRNA entries are written as ID=gene-XXX, so strip the
# "gene-" prefix to match adata.var_names
import re

rrna_genes = set()
trna_genes = set()

with open("ea.gff") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if len(cols) < 9:
            continue
        m = re.search(r"gene_biotype=(rRNA|tRNA)", cols[8])
        if not m:
            continue
        mid = re.search(r"ID=([^;]+)", cols[8])
        if mid:
            gid = mid.group(1).replace("gene-", "")
            (rrna_genes if m.group(1) == "rRNA" else trna_genes).add(gid)

print(f"rRNA genes: {len(rrna_genes)}, tRNA genes: {len(trna_genes)}")

In [ ]:
# cp_all -> run1 barcodes -> remove rRNA/tRNA -> normalize -> UMAP
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1

cp_all = sc.read_h5ad("cp_all.h5ad")
run1_cp = pd.read_csv("run1_cp.barcodes.tsv", sep="\t")

# Keep barcodes present in run1_cp.barcodes.tsv
adata_umap = cp_all[cp_all.obs_names.isin(run1_cp["barcode"])].copy()
adata_umap.obs["call"] = pd.Categorical(
    run1_cp.set_index("barcode").reindex(adata_umap.obs_names)["call"].values,
    categories=["cell", "empty"],
)
print(f"Barcodes: {adata_umap.n_obs:,} "
      f"(cell={(adata_umap.obs['call'] == 'cell').sum():,}, "
      f"empty={(adata_umap.obs['call'] == 'empty').sum():,})")

# Remove rRNA + tRNA genes
drop_mask = adata_umap.var_names.isin(rrna_genes | trna_genes)
adata_umap = adata_umap[:, ~drop_mask].copy()
print(f"Removed {drop_mask.sum()} rRNA/tRNA genes -> {adata_umap.n_vars:,} genes")

# Barcodes with no non-rRNA/tRNA UMI carry no expression signal: report and drop
nonrrna_umi = np.asarray(adata_umap.X.sum(axis=1)).ravel()
zero_mask = nonrrna_umi == 0
print(f"Barcodes with 0 non-rRNA/tRNA UMI: {zero_mask.sum():,} "
      f"(cell={(adata_umap.obs['call'][zero_mask] == 'cell').sum():,}, "
      f"empty={(adata_umap.obs['call'][zero_mask] == 'empty').sum():,}) "
      f"-> excluded from UMAP")
adata_umap = adata_umap[~zero_mask].copy()
adata_umap.obs["nonrrna_umi"] = nonrrna_umi[~zero_mask]
print(adata_umap.obs.groupby("call", observed=True)["nonrrna_umi"].median())

# Normalize -> log1p -> HVG
sc.pp.normalize_total(adata_umap, target_sum=1e4)
sc.pp.log1p(adata_umap)
sc.pp.highly_variable_genes(adata_umap, n_top_genes=2000, flavor="seurat")
print(f"HVG: {adata_umap.var['highly_variable'].sum()}")
adata_umap = adata_umap[:, adata_umap.var["highly_variable"]].copy()

# PCA -> neighbors -> UMAP
sc.pp.pca(adata_umap, n_comps=100, zero_center=False, random_state=0)
sc.pp.neighbors(adata_umap, n_neighbors=50, n_pcs=80, random_state=0)
sc.tl.umap(adata_umap, min_dist=0.5, random_state=0)
print("UMAP done:", adata_umap.shape)

In [ ]:
import matplotlib.pyplot as plt

bc_mag_cp = pd.read_csv("bc_mag_cp.tsv", sep="\t")
sp_all = (
    bc_mag_cp.set_index("barcode")["species"]
    .reindex(adata_umap.obs_names)
    .astype("Int64").astype(str)
    .replace("<NA>", "NA")
)
adata_umap.obs["species"] = pd.Categorical(
    sp_all.values, categories=["400667", "679895", "NA"]
)

colors = {"679895": "#4C72B0", "400667": "#C44E52", "NA": "#BDBDBD"}
labels = {"679895": "E. coli (679895)", "400667": "A. baumannii (400667)", "NA": "unassigned"}
u = adata_umap.obsm["X_umap"]

fig, ax = plt.subplots(figsize=(5.6, 4.6))
for sp in ["NA", "400667", "679895"]:
    sel = (adata_umap.obs["species"] == sp).values
    ax.scatter(u[sel, 0], u[sel, 1], s=0.5, c=colors[sp], linewidths=0,
               rasterized=True, label=f"{labels[sp]} (n={sel.sum():,})")
ax.set_title(f"all barcodes incl. empty (n={len(u):,})", fontsize=10)
ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_xticks([])
ax.set_yticks([])
ax.legend(markerscale=10, frameon=False, fontsize=8, loc="best")
plt.tight_layout()

In [ ]:
cell_bcs = run1_cp.loc[run1_cp["call"] == "cell", "barcode"]
adata_cell = cp_all[cp_all.obs_names.isin(cell_bcs)].copy()

sp_cell = (
    bc_mag_cp.set_index("barcode")["species"]
    .reindex(adata_cell.obs_names)
    .astype("Int64").astype(str)
    .replace("<NA>", "NA")
)
adata_cell.obs["species"] = pd.Categorical(
    sp_cell.values, categories=["400667", "679895", "NA"]
)

# remove rRNA / tRNA
adata_cell = adata_cell[:, ~adata_cell.var_names.isin(rrna_genes | trna_genes)].copy()

# normalize -> log1p -> HVG -> PCA -> neighbors -> UMAP
sc.pp.normalize_total(adata_cell, target_sum=1e4)
sc.pp.log1p(adata_cell)
sc.pp.highly_variable_genes(adata_cell, n_top_genes=2000, flavor="seurat")
adata_cell = adata_cell[:, adata_cell.var["highly_variable"]].copy()
sc.pp.pca(adata_cell, n_comps=100, zero_center=False, random_state=0)
sc.pp.neighbors(adata_cell, n_neighbors=50, n_pcs=80, random_state=0)
sc.tl.umap(adata_cell, min_dist=0.5, random_state=0)
print("cell-only UMAP ready:", adata_cell.shape)

print("cell-only UMAP ready:", adata_cell.shape)

import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

colors = {"679895": "#4C72B0", "400667": "#C44E52", "NA": "#BDBDBD"}
labels = {
    "679895": "E. coli (679895)",
    "400667": "A. baumannii (400667)",
    "NA": "unassigned",
}

fig, axes = plt.subplots(1, 2, figsize=(5.2, 2.3))

# panel 1: cell-only
ax = axes[0]
u = adata_cell.obsm["X_umap"]
n_cell = adata_cell.n_obs
for sp in ["400667", "679895"]:
    sel = (adata_cell.obs["species"] == sp).values
    ax.scatter(u[sel, 0], u[sel, 1], s=1.5, c=colors[sp], linewidths=0,
               rasterized=True, label=f"{labels[sp]} (n={sel.sum():,})")
ax.set_title(f"cell barcodes only (n={n_cell:,})", fontsize=6)
ax.set_xlabel("UMAP1", fontsize=6, labelpad=1.5); ax.set_ylabel("UMAP2", fontsize=6, labelpad=1.5)
ax.tick_params(labelsize=6)
ax.set_xticks([]); ax.set_yticks([])
ax.legend(title=f"species (n={n_cell:,})", markerscale=4, frameon=False,
          fontsize=6, title_fontsize=6, loc="best")

# panel 2: all run1 barcodes (from adata_umap, already embedded with same params in cell 13)
ax = axes[1]
u = adata_umap.obsm["X_umap"]
n_all = adata_umap.n_obs
for sp in ["NA", "400667", "679895"]:
    sel = (adata_umap.obs["species"] == sp).values
    if sel.sum() == 0:
        continue
    ax.scatter(u[sel, 0], u[sel, 1], s=0.5, c=colors[sp], linewidths=0,
               rasterized=True, label=f"{labels[sp]} (n={sel.sum():,})")
ax.set_title(f"all run1 barcodes (n={n_all:,})", fontsize=6)
ax.set_xlabel("UMAP1", fontsize=6, labelpad=1.5); ax.set_ylabel("UMAP2", fontsize=6, labelpad=1.5)
ax.tick_params(labelsize=6)
ax.set_xticks([]); ax.set_yticks([])
ax.legend(title=f"species (n={n_all:,})", markerscale=4, frameon=False,
          fontsize=6, title_fontsize=6, loc="best")

plt.tight_layout()
print("Saved: umap_k50_pcs80_cell_vs_all.pdf/.png")

In [ ]:
# Panel f: UMI distribution step plot — 4 calling methods
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

df = pd.read_csv("run1_cp.barcodes.tsv", sep="\t")

# Method 1: MVAcall (call == cell)
df_md = df[df["call"] == "cell"].copy()

# Method 2: Knee — find elbow on rank-UMI curve
df_sorted = df.sort_values("rank").reset_index(drop=True)
log_umis = np.log10(df_sorted["umis"].clip(lower=1))
x = np.arange(len(log_umis))
# Normalize both dimensions
x_norm = (x - x.mean()) / x.std()
y_norm = (log_umis - log_umis.mean()) / log_umis.std()
# Distance from diagonal y = -x (elbow point)
dist = x_norm + y_norm  # max distance from diagonal
knee_idx = int(np.argmax(dist))
df_knee = df_sorted.iloc[:knee_idx].copy()

# Method 3: EmptyDrops — padj_non_rrna < 0.01
df_ed_nr = df[
    (df["padj_non_rrna"].notna()) & (df["padj_non_rrna"] < 0.01)
].copy()

# Method 4: EmptyDrops — padj_combined < 0.01 (total RNA)
df_ed_total = df[
    (df["padj_combined"].notna()) & (df["padj_combined"] < 0.01)
].copy()

# UMI bins
min_u = min(df_md["umis"].min(), df_knee["umis"].min(),
            df_ed_nr["umis"].min(), df_ed_total["umis"].min())
max_u = max(df_md["umis"].max(), df_knee["umis"].max(),
            df_ed_nr["umis"].max(), df_ed_total["umis"].max())
bins = np.logspace(np.log10(max(min_u, 1)), np.log10(max_u), 30)

def bin_counts(data, bins):
    data = data.copy()
    data["umi_bin"] = pd.cut(data["umis"], bins=bins, include_lowest=True)
    return data.groupby("umi_bin", observed=False).size()

x_step = np.log10(bins)[:-1]

fig, ax = plt.subplots(figsize=(4.2, 3.0))
ax.step(x_step, bin_counts(df_md, bins).values, where="pre", linewidth=1,
        color="#2E86C1", label=f"MVAcall (n={len(df_md):,})")
ax.step(x_step, bin_counts(df_knee, bins).values, where="pre", linewidth=1,
        color="#E74C3C", label=f"Knee (n={len(df_knee):,})")
ax.step(x_step, bin_counts(df_ed_nr, bins).values, where="pre", linewidth=1,
        color="#28B463", label=f"EmptyDrops non-rRNA (n={len(df_ed_nr):,})")
ax.step(x_step, bin_counts(df_ed_total, bins).values, where="pre", linewidth=1,
        color="#8E44AD", label=f"EmptyDrops total RNA (n={len(df_ed_total):,})")

ax.set_xlabel("log10(UMI per barcode)", fontsize=8)
ax.set_ylabel("Number of Cells", fontsize=8)
ax.set_title("Comparison of Cell Calling Methods by UMI Distribution", fontsize=9)
ax.legend(fontsize=6, loc="upper right")
plt.tight_layout()

In [ ]:
import re

rrna_genes = set()

with open("ea.gff") as f:
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip().split("\t")
        if len(cols) < 9:
            continue
        if "gene_biotype=rRNA" in cols[8]:
            m = re.search(r"ID=([^;]+)", cols[8])
            if m:
                rrna_genes.add(m.group(1))

print(f"rRNA genes: {len(rrna_genes)}")

In [ ]:
# MD vs ED (emptydrops_total_rna) cell calling comparison (cp)
# MD: cp.md.tsv (call=='cell' in run1_cp.barcodes.tsv)
# ED: cp.ed.tsv (padj_non_rrna < 0.01 in run1_cp.barcodes.tsv)

import pandas as pd

cp_md = pd.read_csv('cp.md.tsv', sep='\t')
cp_ed = pd.read_csv('cp.ed.tsv', sep='\t')
bc_mag_cp = pd.read_csv('bc_mag_cp.tsv', sep='\t')

# Per-cell stats from each matrix
def cell_stats(df, prefix):
    nonrrna = df[~df['gene'].isin(rrna_genes)]
    total = df.groupby('cell')['count'].agg(['sum','count']).reset_index()
    total.columns = ['barcode', f'{prefix}_umi', f'{prefix}_gene']
    nr = nonrrna.groupby('cell')['count'].agg(['sum','count']).reset_index()
    nr.columns = ['barcode', f'{prefix}_nonrrna_umi', f'{prefix}_nonrrna_gene']
    return total.merge(nr, on='barcode')

md_stats = cell_stats(cp_md, 'md')
ed_stats = cell_stats(cp_ed, 'ed')

# Merge all
df_cmp = md_stats.merge(ed_stats, on='barcode', how='outer')
df_cmp = df_cmp.merge(bc_mag_cp[['barcode','purity']], on='barcode', how='left')

# Group assignment
md_cells = set(cp_md['cell'].unique())
ed_cells = set(cp_ed['cell'].unique())
df_cmp['group'] = 'neither'
df_cmp.loc[df_cmp['barcode'].isin(md_cells & ed_cells), 'group'] = 'overlap'
df_cmp.loc[(df_cmp['barcode'].isin(md_cells)) & (~df_cmp['barcode'].isin(ed_cells)), 'group'] = 'MD_only'
df_cmp.loc[(~df_cmp['barcode'].isin(md_cells)) & (df_cmp['barcode'].isin(ed_cells)), 'group'] = 'ED_only'

# Unified metrics: use md stats for overlap/MD_only, ed stats for ED_only
df_cmp['umi'] = df_cmp['md_umi'].fillna(df_cmp['ed_umi'])
df_cmp['gene'] = df_cmp['md_nonrrna_gene'].fillna(df_cmp['ed_nonrrna_gene'])
df_cmp['nonrrna_umi'] = df_cmp['md_nonrrna_umi'].fillna(df_cmp['ed_nonrrna_umi'])
df_cmp['nonrrna_gene'] = df_cmp['md_nonrrna_gene'].fillna(df_cmp['ed_nonrrna_gene'])

print(f'MD called: {len(md_cells)} cells')
print(f'ED called: {len(ed_cells)} cells')
print(f'Overlap: {len(md_cells & ed_cells)}')
print(f'MD only: {len(md_cells - ed_cells)}')
print(f'ED only: {len(ed_cells - md_cells)}')

print('\n' + '=' * 90)
print(f"{'Group':<12} {'n':>6}  {'umi_med':>10} {'gene_med':>10} {'nonrrna_umi_med':>16} {'purity_med':>12} {'purity>=0.9':>12}")
print('-' * 90)
for g in ['overlap', 'MD_only', 'ED_only']:
    sub = df_cmp[df_cmp['group'] == g]
    n = len(sub)
    umi_m = sub['umi'].median()
    gene_m = sub['gene'].median()
    nr_umi_m = sub['nonrrna_umi'].median()
    pur_m = sub['purity'].median()
    n_pur = sub['purity'].notna().sum()
    n_high = (sub['purity'] >= 0.9).sum()
    pct = f'{n_high}/{n_pur} ({n_high/n_pur*100:.1f}%)' if n_pur > 0 else '-'
    print(f'{g:<12} {n:>6}  {umi_m:>10.0f} {gene_m:>10.0f} {nr_umi_m:>16.0f} {pur_m:>12.3f} {pct:>12}')
print('=' * 90)

In [ ]:
# Violin plots: Non-rRNA UMIs, Non-rRNA genes by calling group

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats as sp_stats

group_order = ['overlap', 'MD_only', 'ED_only']
group_colors = {'overlap': '#4C72B0', 'MD_only': '#55A868', 'ED_only': '#C44E52'}
group_labels = {
    'overlap': f'Overlap\n(n={(df_cmp["group"]=="overlap").sum()})',
    'MD_only': f'MD only\n(n={(df_cmp["group"]=="MD_only").sum()})',
    'ED_only': f'ED only\n(n={(df_cmp["group"]=="ED_only").sum()})',
}

plot_metrics = [
    ('nonrrna_umi', 'Non-rRNA UMIs'),
    ('nonrrna_gene', 'Non-rRNA genes'),
]

fig, axes = plt.subplots(1, 2, figsize=(4.8, 2.4))

for ax, (metric, title) in zip(axes, plot_metrics):
    data = []
    for g in group_order:
        vals = df_cmp[df_cmp['group'] == g][metric].dropna().values
        data.append(vals)

    parts = ax.violinplot(data, positions=range(3), showmedians=True, showextrema=False)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(group_colors[group_order[i]])
        pc.set_alpha(0.5)
    ax.tick_params(labelsize=6, width=0.6, length=2.2)
    for sp_ in ('top','right'):
        ax.spines[sp_].set_visible(False)
        ax.spines[sp_].set_linewidth(0.6)
    if 'cmedians' in parts:
        parts['cmedians'].set_color('black')

    ax.set_xticks(range(3))
    ax.set_xticklabels([group_labels[g] for g in group_order], fontsize=6)
    ax.set_title(title, fontsize=6)

    # Mann-Whitney U: ED_only vs overlap
    ed_vals = df_cmp[df_cmp['group'] == 'ED_only'][metric].dropna().values
    ov_vals = df_cmp[df_cmp['group'] == 'overlap'][metric].dropna().values
    if len(ed_vals) > 0 and len(ov_vals) > 0:
        u_stat, p_val = sp_stats.mannwhitneyu(ed_vals, ov_vals, alternative='two-sided')
        ax.text(0.5, 0.95, f'ED vs overlap\np={p_val:.2e}',
                transform=ax.transAxes, ha='center', va='top', fontsize=6,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
print('Saved: md_vs_ed_violin.pdf')